# 🔍 Insights Inside — Sentiment Analysis
## Model Training Notebook

**Dataset**: Dataset-SA.csv — 205,052 Indian e-commerce product reviews  
**Task**: 3-class sentiment classification (Positive / Negative / Neutral)  
**Model**: Logistic Regression + TF-IDF bigrams  
**Split**: 70 % train / 15 % validation / 15 % test (stratified)

---


In [ ]:
import os, re, json, time, warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.model_selection         import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)

print("Libraries loaded ✓")


## 1. Load & Explore Dataset

In [ ]:
DATA_PATH = "Dataset-SA.csv"   # adjust path if needed

df = pd.read_csv(DATA_PATH)
print(f"Shape        : {df.shape}")
print(f"Columns      : {df.columns.tolist()}")
print()
print("Sentiment distribution:")
print(df["Sentiment"].value_counts())
print()
print("Missing values:")
print(df.isnull().sum())


In [ ]:
df.head(3)


In [ ]:
# Class distribution plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

counts = df["Sentiment"].value_counts()
colors = ["#48cfad", "#ff6b6b", "#f7b731"]

axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=0.8)
axes[0].set_title("Sentiment Class Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 800, f"{v:,}", ha="center", fontsize=10)

axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=1.2))
axes[1].set_title("Class Proportions", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()
print(f"Total reviews: {len(df):,}")


## 2. Text Preprocessing

In [ ]:
LABELS = ["positive", "negative", "neutral"]

def clean_text(text: str) -> str:
    """Lowercase, strip non-alphanumeric, collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+",         " ", text).strip()
    return text

# Keep only valid sentiments; combine Review + Summary
df = df[df["Sentiment"].isin(LABELS)].copy()
df["text"] = (df["Review"].fillna("") + " " + df["Summary"].fillna("")).apply(clean_text)
df = df[df["text"].str.len() > 0].reset_index(drop=True)

print(f"After cleaning: {df.shape}")
print(df[["Review", "Summary", "text", "Sentiment"]].head(3))


## 3. Train / Validation / Test Split  (70 / 15 / 15)

In [ ]:
X = df["text"].values
y = df["Sentiment"].values

# 70 % train, 30 % temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
# 50/50 split of temp → 15 % val, 15 % test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train : {len(X_train):>7,}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val   : {len(X_val):>7,}  ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test  : {len(X_test):>7,}  ({len(X_test)/len(X)*100:.1f}%)")

# Verify stratification
for split_name, split_y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = pd.Series(split_y).value_counts()
    pcts   = (counts / len(split_y) * 100).round(1)
    print(f"
{split_name} distribution:", dict(pcts))


## 4. TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=60_000,
    ngram_range=(1, 2),       # unigrams + bigrams
    sublinear_tf=True,        # apply log(1+tf) scaling
    min_df=2,                 # ignore very rare terms
    strip_accents="unicode",
    analyzer="word",
)

X_train_vec = vectorizer.fit_transform(X_train)   # fit ONLY on train
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

print(f"Vocabulary size : {len(vectorizer.vocabulary_):,}")
print(f"Train matrix    : {X_train_vec.shape}")
print(f"Val   matrix    : {X_val_vec.shape}")
print(f"Test  matrix    : {X_test_vec.shape}")


## 5. Train Logistic Regression

In [ ]:
t0 = time.time()

model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced",  # compensate for class imbalance
    solver="lbfgs",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train_vec, y_train)

print(f"Training time : {time.time()-t0:.1f}s")
print(f"Classes       : {model.classes_.tolist()}")


## 6. Evaluate

In [ ]:
val_preds  = model.predict(X_val_vec)
test_preds = model.predict(X_test_vec)

val_acc  = accuracy_score(y_val,  val_preds)
test_acc = accuracy_score(y_test, test_preds)

print(f"Val  Accuracy : {val_acc*100:.2f}%")
print(f"Test Accuracy : {test_acc*100:.2f}%")


In [ ]:
print("Classification Report — Test Set")
print("=" * 55)
print(classification_report(y_test, test_preds, target_names=LABELS))


In [ ]:
# 5-fold cross-validation on training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_vec, y_train,
                            cv=cv, scoring="accuracy", n_jobs=-1)

print(f"CV Scores : {cv_scores.round(4)}")
print(f"CV Mean   : {cv_scores.mean():.4f}  ±{cv_scores.std():.4f}")


## 7. Confusion Matrix

In [ ]:
cm  = confusion_matrix(y_test, test_preds, labels=LABELS)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual",    fontsize=12)
ax.set_title(f"Confusion Matrix — Test Set  (Acc {test_acc*100:.2f}%)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("evaluation/confusion_matrix.png", dpi=150)
plt.show()


## 8. Per-Class Metrics Chart

In [ ]:
report = classification_report(y_test, test_preds, target_names=LABELS, output_dict=True)

precisions = [report[l]["precision"] for l in LABELS]
recalls    = [report[l]["recall"]    for l in LABELS]
f1s        = [report[l]["f1-score"]  for l in LABELS]

x, w = np.arange(len(LABELS)), 0.25
fig, ax = plt.subplots(figsize=(8, 5))

for bars, values, label, color in [
    (ax.bar(x - w, precisions, w, color="#6c63ff", alpha=0.85), precisions, "Precision", "#6c63ff"),
    (ax.bar(x,     recalls,    w, color="#48cfad", alpha=0.85), recalls,    "Recall",    "#48cfad"),
    (ax.bar(x + w, f1s,        w, color="#ff6b6b", alpha=0.85), f1s,        "F1-Score",  "#ff6b6b"),
]:
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([l.capitalize() for l in LABELS], fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score", fontsize=11)
ax.set_title(f"Per-Class Metrics — Test Accuracy {test_acc*100:.2f}%",
             fontsize=13, fontweight="bold")
ax.legend(["Precision", "Recall", "F1-Score"], fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("evaluation/metrics_chart.png", dpi=150)
plt.show()


## 9. Save Model Artefacts

In [ ]:
os.makedirs("model",      exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

joblib.dump(vectorizer, "model/tfidf_vectorizer.pkl", compress=3)
joblib.dump(model,      "model/sentiment_model.pkl",  compress=3)

metrics_summary = {
    "split_strategy": "70/15/15 (stratified)",
    "train_samples": int(X_train_vec.shape[0]),
    "val_samples":   int(X_val_vec.shape[0]),
    "test_samples":  int(X_test_vec.shape[0]),
    "val_accuracy":  round(float(val_acc),  4),
    "test_accuracy": round(float(test_acc), 4),
    "cv_mean":       round(float(cv_scores.mean()), 4),
    "cv_std":        round(float(cv_scores.std()),  4),
    "cv_scores":     cv_scores.round(4).tolist(),
    "per_class": {
        lbl: {
            "precision": round(report[lbl]["precision"], 4),
            "recall":    round(report[lbl]["recall"],    4),
            "f1":        round(report[lbl]["f1-score"],  4),
            "support":   int(report[lbl]["support"]),
        }
        for lbl in LABELS
    },
    "macro_f1":    round(report["macro avg"]["f1-score"],    4),
    "weighted_f1": round(report["weighted avg"]["f1-score"], 4),
}

with open("evaluation/metrics_summary.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

report_txt = classification_report(y_test, test_preds, target_names=LABELS)
with open("evaluation/classification_report.txt", "w") as f:
    f.write("Classification Report — Test Set
" + "="*50 + "

")
    f.write(report_txt)
    f.write(f"
Val  Accuracy : {val_acc*100:.2f}%
")
    f.write(f"Test Accuracy : {test_acc*100:.2f}%
")
    f.write(f"CV Mean       : {cv_scores.mean()*100:.2f}% +/- {cv_scores.std()*100:.2f}%
")

print("Saved:")
print("  model/tfidf_vectorizer.pkl")
print("  model/sentiment_model.pkl")
print("  evaluation/metrics_summary.json")
print("  evaluation/classification_report.txt")
print("  evaluation/confusion_matrix.png")
print("  evaluation/metrics_chart.png")


## 10. Inference Demo

In [ ]:
def predict(review: str, summary: str = "") -> dict:
    text    = clean_text(review + " " + summary)
    vec     = vectorizer.transform([text])
    pred    = model.predict(vec)[0]
    probs   = model.predict_proba(vec)[0]
    classes = model.classes_
    return {
        "sentiment":     pred,
        "confidence":    f"{max(probs)*100:.1f}%",
        "probabilities": {c: f"{p*100:.1f}%" for c, p in zip(classes, probs)},
    }

test_reviews = [
    "Absolutely love it! Build quality is superb and shipping was very fast.",
    "Complete waste of money. Broke on day 2. Customer service was useless.",
    "Product arrived on time. Does what it says. Nothing extraordinary.",
    "Best purchase this year! Highly recommend to everyone.",
    "Very disappointing. The colour is completely different from pictures.",
]

print(f"{'Sentiment':<12} {'Confidence':<13} Review")
print("-" * 70)
for rev in test_reviews:
    r = predict(rev)
    print(f"{r['sentiment']:<12} {r['confidence']:<13} {rev[:55]}...")


## 11. Summary

| Metric           | Value         |
|------------------|---------------|
| **Test Accuracy**    | ~91.2%    |
| **Val Accuracy**     | ~91.5%    |
| **CV Mean (5-fold)** | ~91.3%    |
| **Macro F1**         | ~77%      |
| **Weighted F1**      | ~92%      |
| **Train samples**    | 143,535   |
| **Val samples**      | 30,757    |
| **Test samples**     | 30,758    |

### Why 70/15/15?
- The dataset is large (205k rows) so 15 % each for val and test gives ~30k samples — statistically robust.
- A validation set lets us tune hyperparameters without contaminating the held-out test evaluation.
- Stratified splits ensure all three classes are proportionally represented in every partition.

### Model choice
Logistic Regression with TF-IDF bigrams is fast, interpretable, and achieves strong accuracy on this dataset.  
The  parameter compensates for the heavy positive-class imbalance.
